In [1]:
!pip install -U langchain-community langchain-openai faiss-cpu pypdf python-docx docx2txt openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 951.0/951.0 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: openai
    Found existing installation: openai 1.106.1
    Uninstalling openai-1.106.1:
      Successfully uninstalled openai-1.106.1
ERROR: pip's dependency resolver does not curr

In [ ]:
!pip install --upgrade openai python-dateutil >/dev/null

import os, json, time, uuid
from typing import Any, Dict
from openai import OpenAI
from dateutil.parser import parse as parse_date, ParserError
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# =====================================================
# 🔑 API Key + Client
# =====================================================
OPENAI_API_KEY = ""   # <<< PUT YOUR KEY HERE
client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = "gpt-4.1-mini"

# =====================================================
# 📂 Paths
# =====================================================
FOLDER_PATH = "/content/wedlii_docs"
INDEX_PATH = "/content/faiss_index"
USERS_FILE = "users.json"
CONVOS_FILE = "conversations.json"
CONVO_BUFFER_SIZE = 10

# =====================================================
# 🔎 FAISS Index
# =====================================================
embedding = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

if os.path.exists(f"{INDEX_PATH}/index.faiss"):
    print("✅ Loading existing FAISS index...")
    vectorstore = FAISS.load_local(INDEX_PATH, embedding, allow_dangerous_deserialization=True)
else:
    if not os.path.exists(FOLDER_PATH):
        os.makedirs(FOLDER_PATH)
        print(f"📂 Created '{FOLDER_PATH}'. Please add Wedlii PDF files and rerun.")
        raise SystemExit

    all_documents = []
    for file in os.listdir(FOLDER_PATH):
        if not file.lower().endswith(".pdf"):
            print(f"⚠️ Skipping non-PDF file: {file}")
            continue
        try:
            loader = PyPDFLoader(os.path.join(FOLDER_PATH, file))
            docs = loader.load()
            for i, doc in enumerate(docs, 1):
                doc.metadata.update({
                    "source_file": file,
                    "chunk_number": i,
                    "page_number": i
                })
            all_documents.extend(docs)
            print(f"✅ Loaded {len(docs)} pages from {file}")
        except Exception as e:
            print(f"❌ Error loading {file}: {e}")

    if not all_documents:
        print("⚠️ No PDFs found. Exiting.")
        raise SystemExit

    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    chunks = splitter.split_documents(all_documents)

    vectorstore = FAISS.from_documents(chunks, embedding)
    vectorstore.save_local(INDEX_PATH)
    print("✅ FAISS index created and saved!")

# =====================================================
# 🗂️ JSON Storage Helpers
# =====================================================
def safe_load_json(path: str) -> Dict[str, Any]:
    if not os.path.exists(path): return {}
    try:
        with open(path, "r", encoding="utf-8") as f: return json.load(f)
    except: return {}

def safe_write_json(path: str, data: Dict[str, Any]):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f: json.dump(data, f, indent=2, ensure_ascii=False)
    os.replace(tmp, path)

users_store = safe_load_json(USERS_FILE)
convos_store = safe_load_json(CONVOS_FILE)
if not os.path.exists(USERS_FILE): safe_write_json(USERS_FILE, users_store)
if not os.path.exists(CONVOS_FILE): safe_write_json(CONVOS_FILE, convos_store)

# =====================================================
# 👤 Default User
# =====================================================
def default_user(user_id: str) -> Dict[str, Any]:
    return {
        "user_id": user_id,
        "first_name": "",
        "last_name": "",
        "wedding_date": "",
        "venue": "",
        "celebrant": "",
        "style": "",
        "guest_count": None,
        "total_budget": None,
        "memory_summary": ""  # 🧠 Long-term memory
    }

# =====================================================
# 📝 Conversation Handling
# =====================================================
def load_conversation(user_id: str) -> list:
    return convos_store.get(user_id, [])

def append_conversation_turn(user_id: str, user_msg: str, assistant_msg: str):
    history = convos_store.get(user_id, [])
    history.append({"user_message": user_msg, "assistant_message": assistant_msg, "ts": int(time.time())})
    history = history[-CONVO_BUFFER_SIZE:]
    convos_store[user_id] = history
    safe_write_json(CONVOS_FILE, convos_store)

# =====================================================
# 🧠 Summarized Memory
# =====================================================
def summarize_memory(user_id: str):
    history = convos_store.get(user_id, [])
    if not history:
        return ""
    conversation_text = "\n".join([
        f"User: {turn['user_message']}\nBot: {turn['assistant_message']}"
        for turn in history
    ])
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "Summarize the conversation into key facts about the user, preferences, and wedding details."},
                {"role": "user", "content": conversation_text}
            ],
            max_tokens=200
        )
        summary = resp.choices[0].message.content.strip()

        # ✅ Ensure user exists
        if user_id not in users_store:
            users_store[user_id] = default_user(user_id)

        users_store[user_id]["memory_summary"] = summary
        safe_write_json(USERS_FILE, users_store)
    except Exception as e:
        print("⚠️ Error summarizing memory:", e)

# =====================================================
# System Prompt Builder
# =====================================================
def build_system_prompt(user_id: str) -> str:
    user = users_store.get(user_id, default_user(user_id))
    details = "\n".join([
        f"{k}: {user[k]}" for k in ["wedding_date","venue","celebrant","style","guest_count","total_budget"]
    ])
    memory = user.get("memory_summary", "")
    return f"""You are Wedlii, a warm and caring wedding planner.
Always be concise and supportive.

User details:
{details}

Memory of past chats:
{memory}
"""

# =====================================================
# User Detail Functions
# =====================================================
def validate_input(field: str, value: str):
    try:
        if field == "wedding_date":
            parse_date(value)
            return value
        if field == "guest_count":
            return int(value)
        if field == "total_budget":
            return float(value)
        return value
    except (ValueError, ParserError):
        return None

def update_user_detail(user_id: str, field: str, value: Any):
    if user_id not in users_store:
        users_store[user_id] = default_user(user_id)
    validated = validate_input(field, value)
    if validated is None:
        return {"error": f"Invalid value for {field}: {value}"}
    users_store[user_id][field] = validated
    safe_write_json(USERS_FILE, users_store)
    return users_store[user_id]

def get_user_detail(user_id: str, field: str):
    return users_store.get(user_id, {}).get(field)

# =====================================================
# Main Chat Logic
# =====================================================
def ask_wedlii(user_id: str, user_text: str, temperature: float = 0.5):
    # Ensure user exists
    if user_id not in users_store:
        users_store[user_id] = default_user(user_id)
        safe_write_json(USERS_FILE, users_store)

    system_prompt = build_system_prompt(user_id)
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_text}
    ]

    # Add FAISS context
    try:
        docs = vectorstore.similarity_search(user_text, k=2)
        context = "\n\n".join([doc.page_content for doc in docs])
        if context:
            messages.append({"role": "system", "content": f"Knowledge Base Context:\n{context}"})
    except Exception:
        pass

    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            max_tokens=250,
            temperature=temperature
        )
    except Exception as e:
        return f"⚠️ Error contacting AI: {e}", None

    msg = resp.choices[0].message
    assistant_msg = (msg.content or "").strip()

    append_conversation_turn(user_id, user_text, assistant_msg)
    summarize_memory(user_id)

    return assistant_msg, resp

# =====================================================
# CLI
# =====================================================
print("Welcome to Wedlii Assistant! Enter user ID or press enter for new one.")
uid = input("User ID: ").strip()
if not uid:
    uid = str(uuid.uuid4())
    users_store[uid] = default_user(uid)
    safe_write_json(USERS_FILE, users_store)
    print(f"✨ New user created: {uid}")
else:
    if uid not in users_store:
        users_store[uid] = default_user(uid)
        safe_write_json(USERS_FILE, users_store)
        print(f"✨ New profile created for user {uid}")

print("\nChat with Wedlii (type 'exit' to quit).\n")
while True:
    q = input("You: ").strip()
    if q.lower() in ("exit","quit"):
        print("👋 Goodbye!")
        break
    a, _ = ask_wedlii(uid, q)
    print("Bot:", a)
